## Setup (5 pts)

In [ ]:
# Run once:
!pip install openai pdfplumber pdf2image pillow pandas
!pip install -U brotli

  Attempting uninstall: brotli
    Found existing installation: Brotli 1.0.9
    Uninstalling Brotli-1.0.9:
      Successfully uninstalled Brotli-1.0.9


  You can safely remove it manually.


In [ ]:
import os
import base64
import io
from difflib import SequenceMatcher

import pandas as pd
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path
from openai import OpenAI
import getpass

# API Configuration
# Enter your AvalAI API key when prompted.
# The API key is requested securely and is not hardcoded.

api_key = getpass.getpass("Enter your AvalAI API Key: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.avalai.ir/v1"
)
# Cheap/fast default; used for most calls in this notebook.
MODEL_FAST = "gpt-4o-mini"
# Stronger vision quality; use this for Part 3.
MODEL_VISION = "gpt-4o"

SAMPLE_PDF = "sample_field_report.pdf"


In [2]:
# Sanity check: confirm the sample PDF is where we expect it and how many pages it has
with pdfplumber.open(SAMPLE_PDF) as pdf:
    print(f"{SAMPLE_PDF}: {len(pdf.pages)} pages")


sample_field_report.pdf: 3 pages


## Part 1 — LLM API fundamentals (20 pts)

Implement `call_llm` below, then use it for the three exploratory calls that follow.

In [3]:
def call_llm(prompt, system=None, model=MODEL_FAST, max_output_tokens=1024, return_raw=False):
    """Call the LLM and return its text response as a string
    (or the raw response object if return_raw=True)."""
    kwargs = dict(model=model, input=prompt, max_output_tokens=max_output_tokens)
    if system:
        kwargs["instructions"] = system
    response = client.responses.create(**kwargs)
    return response if return_raw else response.output_text

In [ ]:
# 1) A basic call, no system prompt
print(call_llm("What is a PDF, in two sentences?"))
print("-" * 60)



# 2) The same kind of call, but with a system prompt that changes the style/role

print(call_llm(
    "What is a PDF, in two sentences?",
    system="You are a pirate captain. Answer everything in pirate speak.",
))
print("-" * 60)

# 3) A call where you deliberately set max_output_tokens very low for a request
#    that wants a long answer. Print the text, plus response.status and
#    response.incomplete_details, so you can see what actually happened (you'll
#    likely need to call the API directly here, or extend call_llm to optionally
#    return the raw response object instead of just the text).
resp = call_llm(
    "Write a detailed 500-word essay about the history of the PDF format.",
    max_output_tokens=20,
    return_raw=True,
)
print("TEXT:", resp.output_text)
print("STATUS:", resp.status)
print("INCOMPLETE DETAILS:", resp.incomplete_details)


A PDF (Portable Document Format) is a file format developed by Adobe that captures and displays documents in a manner independent of application software, hardware, and operating systems. It retains formatting, fonts, images, and graphics, making it suitable for sharing and printing documents consistently across different devices.
------------------------------------------------------------
Arrr, a PDF be a type of treasure chest for documents, keepin' 'em safe and sound in a format that be viewable on any ship's screen. It be just like a map, holdin' all the secrets without lettin' 'em change upon the sea!
------------------------------------------------------------
TEXT: The Portable Document Format (PDF) has become a cornerstone of document sharing and preservation since its inception in
STATUS: incomplete
INCOMPLETE DETAILS: IncompleteDetails(reason='max_output_tokens')


## Part 2 — Traditional PDF extraction (20 pts)

In [ ]:
# Extract text and tables from each PDF page using pdfplumber.
def extract_text_by_page(pdf_path):
    """Return a list of strings, one per page, using pdfplumber's text extraction."""
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texts.append(page.extract_text() or "")   # None"" :for blank/image-only pages
    return texts


def extract_tables_by_page(pdf_path):
    """Return a list (one entry per page) of that page's extracted tables
    (each table is a list of rows, each row a list of cell strings)."""
    all_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            all_tables.append(page.extract_tables())
    return all_tables


In [6]:
texts = extract_text_by_page(SAMPLE_PDF)
tables = extract_tables_by_page(SAMPLE_PDF)

for i, (t, tb) in enumerate(zip(texts, tables), start=1):
    n_chars = len(t) if t else 0
    print(f"Page {i}: {n_chars} chars, {len(tb)} table(s)")


Page 1: 783 chars, 0 table(s)
Page 2: 332 chars, 1 table(s)
Page 3: 45 chars, 0 table(s)


## Part 3 — Multimodal LLM extraction (30 pts)

### 3a — Build the pipeline yourself (15 pts)

In [ ]:
# Render individual PDF pages as images for vision-based processing.
def render_page_to_image(pdf_path, page_number, dpi=150):
    """Render a single PDF page (1-indexed) to a PIL Image."""
    # page_number, dpi=dpi, and return the single resulting image.
    images = convert_from_path(pdf_path, dpi=dpi, first_page=page_number, last_page=page_number)
    return images[0]


def image_to_base64(img, fmt="PNG"):
    """Given helper: encode a PIL image as a base64 string."""
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

EXTRACTION_PROMPT = (
    "Transcribe this document page faithfully. "
    "1) Output all text content in reading order. "
    "2) If the page has a table, reproduce it as a Markdown table. "
    "3) If part of the page is a scanned image (e.g. a memo), transcribe it too and "
    "mark it with the label [SCANNED CONTENT]. "
    "Do not summarize, do not invent anything that is not visible."
)


def extract_with_vision(image, model=MODEL_VISION, media_type="image/png"):
    """Send a page image to a vision-capable model and return its structured
    transcription of the page (ask for the text content and note any tables)."""
    b64 = image_to_base64(image)
    response = client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_image", "image_url": f"data:{media_type};base64,{b64}"},
                {"type": "input_text", "text": EXTRACTION_PROMPT},
            ],
        }],
        max_output_tokens=2000,
    )
    return response.output_text


In [8]:
def render_page_to_image(pdf_path, page_number, dpi=150):
    images = convert_from_path(
        pdf_path, dpi=dpi,
        first_page=page_number, last_page=page_number,
        poppler_path=r"C:\Users\poppler-26.09.0\Library\bin",
    )
    return images[0]

In [9]:
vision_results = {}
for page_num in range(1, len(texts) + 1):
    img = render_page_to_image(SAMPLE_PDF, page_num)
    vision_results[page_num] = extract_with_vision(img)
    print(f"--- Page {page_num} (vision) ---")
    print(vision_results[page_num][:400])
    print()


--- Page 1 (vision) ---
# Transcription of the Document

---

**GreenLeaf Analytics**

# Q3 Regional Field Report

This report summarizes GreenLeaf Analytics' third-quarter field operations across our four regional distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the Riverbend and Lakeside facilities. Staffing levels remained steady, though two sites reported temporary sl

--- Page 2 (vision) ---
### Transcription of the document page:

---

**Regional Performance - Q3**

| Region     | Q1 Units | Q2 Units | Q3 Units | QoQ Growth |
|------------|----------|----------|----------|------------|
| Riverbend  | 12,400   | 13,100   | 15,850   | +21.0%     |
| Lakeside   | 9,800    | 10,050   | 11,760   | +17.0%     |
| Northgate  | 14,200   | 13,900   | 13,650   | -1.8%      |
| Fairview   | 7,3

--- Page 3 (vision) ---
```
Appendix A: Field Memo (scanned, as received)

[SCANNED CONTENT]

______________________________________________________

### 3b — Use native PDF support (15 pts)

OpenAI's vision-capable models can also accept a PDF directly as an `input_file`
content item, instead of you rendering pages to images yourself — the API extracts
both the text and page images internally. 

In [ ]:
# Send the complete PDF directly to the model without manual image conversion.
def extract_with_native_pdf(pdf_path, model=MODEL_VISION, prompt=None):
    """Send the whole PDF to the model using native PDF support and return its
    response text."""
    # call whose input content list has an "input_file" item (filename=..., and
    # file_data=f"data:application/pdf;base64,{...}") followed by an "input_text"
    # item with your extraction prompt. Return response.output_text.
    if prompt is None:
        prompt = (
        "Extract the content of this PDF page by page. For each page give the text, "
        "reproduce any table as a Markdown table, and transcribe any scanned/image "
        "content too. Do not summarize."
        )
    with open(pdf_path, "rb") as f:
        pdf_b64 = base64.standard_b64encode(f.read()).decode("utf-8")
    response = client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(pdf_path),
                    "file_data": f"data:application/pdf;base64,{pdf_b64}",
                },
                {"type": "input_text", "text": prompt},
            ],
        }],
        max_output_tokens=3000,
    )
    return response.output_text


In [10]:
native_result = extract_with_native_pdf(SAMPLE_PDF)
print(native_result)

### Page 1 / 3

```
GreenLeaf Analytics  
Q3 Regional Field Report  
This report summarizes GreenLeaf Analytics' third-quarter field operations across our four regional  
distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the  
Riverbend and Lakeside facilities. Staffing levels remained steady, though two sites reported temporary  
slowdowns tied to equipment maintenance in August. The sections below break down performance by  
region and flag any open items that require follow-up before the Q4 planning cycle begins.  
Section 2 (next page) presents the regional sales and throughput table referenced above. Section 3 is a  
scanned field memo from the Riverbend site supervisor, included as-received, describing an inventory  
discrepancy that is still under review.  
```

---

### Page 2 / 3

```
Regional Performance - Q3
```

| Region     | Q1 Units | Q2 Units | Q3 Units | QoQ Growth |
|------------|----------|----------|----------|-------

## Part 4 — Compare and analyze (25 pts)

In [ ]:
# Compare traditional extraction and vision-based extraction results.
comparison_rows = []
notes = {
    1: "Traditional extraction captured substantial text from the digital page; no table was detected. Vision also extracted the page content.",
    2: "Traditional extracted text and detected one table. Vision also reconstructed the table, providing a second way to capture the structured content.",
    3: "The scanned memo is not available as ordinary PDF text. Traditional extraction returned only 45 characters, while Vision transcribed the memo content.",
}
for i in range(1, len(texts) + 1):
    comparison_rows.append({
        "page": i,
        "traditional_chars": len(texts[i - 1]),
        "traditional_tables": len(tables[i - 1]),
        "vision_chars": len(vision_results[i]),
        "notes": notes.get(i, ""),
    })
# ... populate comparison_rows ...
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,page,traditional_chars,traditional_tables,vision_chars,notes
0,1,783,0,839,Traditional extraction captured substantial te...
1,2,332,1,566,Traditional extracted text and detected one ta...
2,3,45,0,628,The scanned memo is not available as ordinary ...


## a hybrid pipeline

Implement a hybrid pipeline: use the traditional parser by default, and only fall back to
the (more expensive) vision pipeline when the extracted text for a page looks too short
or empty to be real.

In [ ]:
# Hybrid Pipeline
# Use traditional extraction by default.
# Fall back to vision-based extraction when extracted text is insufficient.
def hybrid_extract(pdf_path, min_chars=100):
    """Return a list of per-page text: traditional extraction where it looks
    sufficient, vision extraction as a fallback otherwise."""
    page_texts = extract_text_by_page(pdf_path)
    results = []
    for i, text in enumerate(page_texts, start=1):
        if text and len(text.strip()) >= min_chars:
            results.append(text)
        else:
            img = render_page_to_image(pdf_path, i)
            results.append(extract_with_vision(img))
    return results


In [13]:
hybrid_results = hybrid_extract(SAMPLE_PDF)

for i, text in enumerate(hybrid_results, start=1):
    source = "traditional" if len(texts[i-1].strip()) >= 100 else "vision"
    print(f"--- Page {i} (source: {source}) ---")
    print(text[:400])
    print()

--- Page 1 (source: traditional) ---
GreenLeaf Analytics
Q3 Regional Field Report
This report summarizes GreenLeaf Analytics' third-quarter field operations across our four regional
distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the
Riverbend and Lakeside facilities. Staffing levels remained steady, though two sites reported temporary
slowdowns tied to equipment maintenance in Augus

--- Page 2 (source: traditional) ---
Regional Performance - Q3
Region Q1 Units Q2 Units Q3 Units QoQ Growth
Riverbend 12,400 13,100 15,850 +21.0%
Lakeside 9,800 10,050 11,760 +17.0%
Northgate 14,200 13,900 13,650 -1.8%
Fairview 7,300 7,410 7,955 +7.4%
Northgate is the only region with negative QoQ growth this quarter; see the open maintenance item noted
in Section 1.

--- Page 3 (source: vision) ---
### Transcription:

---

**Appendix A: Field Memo (scanned, as received)**

---

[SCANNED CONTENT]

GREENLEAF ANALYTICS - FIELD MEMO  
Date: 14 March  |  Site